# GGUF Export Notebook

Use this notebook after fine-tuning has finished and a merged Hugging Face model folder already exists in Google Drive. It only handles GGUF conversion and quantization.

## 1) Mount Google Drive and define paths

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/AIGColab/NLP/FinalProject")
MODELS_DIR = PROJECT_ROOT / "models"
RUN_NAME = "factcheck_lora_20260315_213108"  # e.g. factcheck_lora_20260315_195017
ARTIFACT_DIR = MODELS_DIR / RUN_NAME
MERGED_DIR = ARTIFACT_DIR / "merged"
GGUF_DIR = ARTIFACT_DIR / "gguf"
GGUF_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("MERGED_DIR   =", MERGED_DIR)
print("GGUF_DIR     =", GGUF_DIR)


## 2) Configure GGUF export

In [ ]:
GGUF_F16_NAME = "model-f16.gguf"
GGUF_Q4_NAME = "model-q4_k_m.gguf"
GGUF_QUANT_METHOD = "Q4_K_M"

F16_GGUF_PATH = GGUF_DIR / GGUF_F16_NAME
Q4_GGUF_PATH = GGUF_DIR / GGUF_Q4_NAME


## 3) Validate the merged HF model directory

In [ ]:
required_files = [
    MERGED_DIR / "config.json",
    MERGED_DIR / "tokenizer_config.json",
]
missing = [str(path) for path in required_files if not path.exists()]
if not MERGED_DIR.exists():
    raise FileNotFoundError(f"Merged model directory not found: {MERGED_DIR}")
if missing:
    raise FileNotFoundError(f"Merged model directory is missing required files: {missing}")

print("Merged HF model directory looks valid.")


## 4) Clone and build `llama.cpp`

In [ ]:
LLAMA_CPP_DIR = Path("/content/llama.cpp")
if not LLAMA_CPP_DIR.exists():
    !git clone https://github.com/ggml-org/llama.cpp.git /content/llama.cpp

%cd /content/llama.cpp
!pip -q install -r requirements.txt
!cmake -B build
!cmake --build build --config Release -j 2


## 5) Convert the merged HF model to F16 GGUF

In [ ]:
!python convert_hf_to_gguf.py "{MERGED_DIR}" --outfile "{F16_GGUF_PATH}" --outtype f16

if not F16_GGUF_PATH.exists():
    raise FileNotFoundError(f"F16 GGUF was not produced: {F16_GGUF_PATH}")

print("F16 GGUF:", F16_GGUF_PATH)


## 6) Quantize to Q4_K_M

In [ ]:
!./build/bin/llama-quantize "{F16_GGUF_PATH}" "{Q4_GGUF_PATH}" {GGUF_QUANT_METHOD}

if not Q4_GGUF_PATH.exists():
    raise FileNotFoundError(f"Quantized GGUF was not produced: {Q4_GGUF_PATH}")

print("Q4 GGUF:", Q4_GGUF_PATH)


## 7) Write artifact summary

In [ ]:
import json

artifact_summary = {
    "run_name": RUN_NAME,
    "merged_model_dir": str(MERGED_DIR),
    "gguf_dir": str(GGUF_DIR),
    "f16_gguf": str(F16_GGUF_PATH),
    "q4_gguf": str(Q4_GGUF_PATH),
    "quant_method": GGUF_QUANT_METHOD,
}

with open(GGUF_DIR / "gguf_artifact_summary.json", "w", encoding="utf-8") as f:
    json.dump(artifact_summary, f, indent=2)

print(json.dumps(artifact_summary, indent=2))
